# Driver drowsiness training - Kaggle Lite

Attach datasets containing `DriverDrowsiness_Kaggle_Project.zip`,
`mrlEyes_2018_01.zip`, and `YawDD_Mirror_Kaggle.zip`. Enable a GPU,
then run each code cell from top to bottom.


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, time, zipfile
import torch

print('SETUP STARTED', flush=True)
if not torch.cuda.is_available():
    raise RuntimeError('Enable GPU in Settings > Accelerator and restart the session.')
print('GPU:', torch.cuda.get_device_name(0), flush=True)

INPUT = Path('/kaggle/input')
PROJECT = Path('/kaggle/working/driver_project')
OUTPUT = Path('/kaggle/working/driver_output')
DATA = Path('/kaggle/working/driver_data')
project_zips = list(INPUT.rglob('DriverDrowsiness_Kaggle_Project.zip'))
if len(project_zips) == 1:
    PROJECT.mkdir(parents=True, exist_ok=True)
    print('Extracting project bundle:', project_zips[0], flush=True)
    with zipfile.ZipFile(project_zips[0]) as archive:
        archive.extractall(PROJECT)
elif len(project_zips) == 0:
    scripts = list(INPUT.rglob('scripts/train.py'))
    if len(scripts) != 1:
        raise RuntimeError(f'Attach the project bundle dataset; found {len(scripts)} extracted copies.')
    source = scripts[0].parent.parent
    print('Copying extracted project input:', source, flush=True)
    shutil.copytree(source, PROJECT, dirs_exist_ok=True)
else:
    raise RuntimeError(f'Found {len(project_zips)} project bundles; attach exactly one.')

os.chdir(PROJECT)
os.environ['TORCH_HOME'] = str(PROJECT / 'models/cache')
OUTPUT.mkdir(parents=True, exist_ok=True)
DATA.mkdir(parents=True, exist_ok=True)
print('SETUP COMPLETE:', PROJECT, flush=True)


In [ ]:
print('LOCATING DATASETS', flush=True)
mrl = list(INPUT.rglob('mrlEyes_2018_01.zip'))
if len(mrl) != 1:
    raise RuntimeError(f'Expected one mrlEyes_2018_01.zip, found {len(mrl)}.')
mrl_archive = mrl[0]

yaw_zips = list(INPUT.rglob('YawDD_Mirror_Kaggle.zip'))
if yaw_zips:
    if len(yaw_zips) != 1:
        raise RuntimeError(f'Expected one YawDD_Mirror_Kaggle.zip, found {len(yaw_zips)}.')
    yaw_source = Path('/kaggle/working/yawdd_mirror_videos')
    if not yaw_source.is_dir():
        print('Extracting YawDD Mirror videos...', flush=True)
        with zipfile.ZipFile(yaw_zips[0]) as archive:
            bad = archive.testzip()
            if bad:
                raise RuntimeError(f'Damaged YawDD ZIP member: {bad}')
            archive.extractall(yaw_source)
else:
    videos = [p for p in INPUT.rglob('*') if p.suffix.lower() in {'.avi', '.mp4', '.mov', '.mkv'}
              and 'mirror' in p.as_posix().lower()]
    if not videos:
        raise RuntimeError('Attach YawDD_Mirror_Kaggle.zip or extracted Mirror videos.')
    yaw_source = INPUT
print('MRL:', mrl_archive, flush=True)
print('YawDD:', yaw_source, flush=True)

def run(*args):
    print('RUNNING:', ' '.join(map(str, args)), flush=True)
    process = subprocess.Popen([str(x) for x in args], env=dict(os.environ, PYTHONUNBUFFERED='1'))
    started = time.monotonic()
    while process.poll() is None:
        time.sleep(30)
        print(f'ACTIVE: {(time.monotonic() - started) / 60:.1f} minutes', flush=True)
    if process.returncode:
        raise subprocess.CalledProcessError(process.returncode, process.args)
    print('COMMAND COMPLETE', flush=True)


In [ ]:
print('EYE MODEL START', flush=True)
eye_data, eye_run = DATA / 'eye_state', OUTPUT / 'eye_state'
if not (eye_data / 'metadata.json').is_file():
    run(sys.executable, '-u', 'scripts/prepare_dataset.py', '--source', mrl_archive, '--output', eye_data)
command = [sys.executable, '-u', 'scripts/train.py', '--data', eye_data, '--output', eye_run,
           '--epochs', '15', '--freeze-epochs', '1', '--batch-size', '128',
           '--workers', '2', '--patience', '4', '--device', 'cuda']
if (eye_run / 'last.pt').is_file():
    command.append('--resume')
elif eye_run.exists():
    shutil.rmtree(eye_run)
if not ((eye_run / 'best.pt').is_file() and (eye_run / 'test_metrics.json').is_file()):
    run(*command)
print('EYE MODEL COMPLETE', flush=True)


In [ ]:
print('YAWN MODEL START', flush=True)
yawn_data, yawn_run = DATA / 'yawn', OUTPUT / 'yawn'
if not (yawn_data / 'metadata.json').is_file():
    if yawn_data.exists():
        shutil.rmtree(yawn_data)
    run(sys.executable, '-u', 'scripts/prepare_video_dataset.py',
        '--source', yaw_source, '--output', yawn_data, '--task', 'yawn')
command = [sys.executable, '-u', 'scripts/train_video.py', '--task', 'yawn',
           '--data', yawn_data, '--output', yawn_run, '--epochs', '40',
           '--batch-size', '4', '--workers', '2', '--device', 'cuda']
if (yawn_run / 'last.pt').is_file():
    command.append('--resume')
elif yawn_run.exists():
    shutil.rmtree(yawn_run)
if not ((yawn_run / 'best.pt').is_file() and (yawn_run / 'test_metrics.json').is_file()):
    run(*command)
print('YAWN MODEL COMPLETE', flush=True)


In [ ]:
print('CREATING MODEL BUNDLE', flush=True)
outputs = {'eye_state': eye_run, 'yawn': yawn_run}
bundle = Path('/kaggle/working/trained_models.zip')
with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as archive:
    for task, folder in outputs.items():
        if not (folder / 'test_metrics.json').is_file():
            raise RuntimeError(f'{task} is incomplete: {folder}')
        print(task, json.loads((folder / 'test_metrics.json').read_text()), flush=True)
        for name in ('best.pt', 'config.json', 'test_metrics.json', 'history.json'):
            path = folder / name
            if path.is_file():
                archive.write(path, f'{task}/{name}')
print('DONE:', bundle, flush=True)
